# Track B: GNews MCP behind the APIM gateway

Put the GNews MCP server behind `apim-foundry-{suffix}`, the gateway 05-02 deployed, and enforce the same per-user rules in a gateway policy instead of a server you own. `news-desk-agent` keeps one MCP tool; only its connection changes, to `news-mcp-apim` from 08-05c-01.

The API is a pass-through, and every user sees the same two tools, GNews's own `top_headlines(category, ...)` and `search(q, ...)`. The gateway cannot hide a tool from one user: the tool list comes back in the response body, and Microsoft's APIM documentation says a policy at MCP scope must not read the response body, because [MCP responses are streamed and reading the body breaks the stream](https://learn.microsoft.com/en-us/azure/api-management/monitor-mcp-servers). So the policy decides at call time instead. The policy, [`infra/news-mcp-policy.xml`](infra/news-mcp-policy.xml), validates the Entra token Foundry sends on the user's behalf, reads the JSON-RPC body of each `tools/call`, and answers `top_headlines` for a category the token's roles do not allow, or `search` without `News.Read.All`, with the same denial text as the Track A wrapper. Allowed calls go to GNews with the key from a secret named value, and the user's token stops at the gateway. [08-05c-00](08-05c-00-news-mcp-per-user-access.md) compares the two tracks.

**What this notebook does:**
1. Adds the `news-mcp` MCP API, its policy, the `gnews-mcp` backend and the `gnews-api-key` named value to `apim-foundry-{suffix}`
2. Checks that a request without a token gets 401
3. Creates a `news-desk-agent` version whose tool uses the `news-mcp-apim` connection
4. Signs in user A and user B
5. Runs the test matrix as both users, with the science case set from the roles user B holds now

## Prerequisites

1. **08-05c-01** has run. **08-05c-02 is optional**: this notebook is Track B on its own, and user B then still holds `News.Read.Science`. If 08-05c-02 has run, its Step 8 revoked that role, and the science case in Step 5 expects a denial.
2. **Core gateway**: [05-02](../../05-foundry-project-pattern-setup/05-02-deploy-foundry-core-gateway/05-02-01-deploy-foundry-core-gateway.ipynb) deployed `apim-foundry-{suffix}` in `rg-foundry-core-{suffix}`. This notebook adds one API, its policy, one backend and one named value, and changes nothing else on the gateway.
3. **`GNEWS_API_KEY`** in `.env`: a free key from [gnews.io](https://gnews.io). This notebook makes two or three GNews requests; denied calls make none.
4. **Azure CLI** as in 08-05c-01, with rights to deploy into `rg-foundry-core-{suffix}`.
5. **User B's pass** from 08-05c-01 is valid for 8 hours. If it has expired, re-run Step 5 of 08-05c-01 for a new one.

## Step 0: Set up names, credentials and helpers

Loads `.env`, derives the lab's names from your Azure CLI sign-in (subscription, tenant, suffix, the gateway and the Foundry project), and defines the helpers the later steps use to call Microsoft Graph, ARM, the Azure CLI and Foundry. It only reads: nothing is created yet.

In [2]:
import hashlib, json, os, subprocess, tempfile, time
from pathlib import Path

import requests
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

import news_mcp_checks as checks
import news_mcp_identity as ident
from news_mcp_identity import authz

repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
ENV_FILE = repo_root / '.env'
load_dotenv(ENV_FILE, override=True)

REQUIRED = ['NEWS_MCP_RESOURCE_APP_ID', 'NEWS_MCP_RESOURCE_SP_ID', 'NEWS_DESK_CLIENT_APP_ID', 'NEWS_TEST_USER_UPN',
            'NEWS_TEST_USER_ID', 'NEWS_AGENT_BLUEPRINT_CLIENT_ID', 'NEWS_CONNECTION_API_VERSION', 'GNEWS_API_KEY']
missing = [key for key in REQUIRED if not os.environ.get(key)]
if missing:
    raise SystemExit(f"Missing in .env: {', '.join(missing)}. Run 08-05c-01 first, and add GNEWS_API_KEY.")

account = json.loads(subprocess.run('az account show -o json', shell=True, capture_output=True, text=True).stdout)
SUBSCRIPTION_ID, TENANT_ID = account['id'], account['tenantId']
SUFFIX = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
CHAT_MODEL = os.environ.get('CHAT_MODEL', 'gpt-5.4-mini')
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

RESOURCE_APP_ID = os.environ['NEWS_MCP_RESOURCE_APP_ID']
RESOURCE_SP_ID = os.environ['NEWS_MCP_RESOURCE_SP_ID']
CLIENT_APP_ID = os.environ['NEWS_DESK_CLIENT_APP_ID']
BLUEPRINT_CLIENT_ID = os.environ['NEWS_AGENT_BLUEPRINT_CLIENT_ID']
TEST_USER_ID = os.environ['NEWS_TEST_USER_ID']
CONNECTION_API = os.environ['NEWS_CONNECTION_API_VERSION']

CORE_RG, APIM_NAME = f'rg-foundry-core-{SUFFIX}', f'apim-foundry-{SUFFIX}'
APIM_API_VERSION = '2024-06-01-preview'
APIM = f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{CORE_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}'
LAB_DIR = Path.cwd()

cli = AzureCliCredential()
ARM, FOUNDRY = 'https://management.azure.com/.default', 'https://ai.azure.com/.default'
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cli, allow_preview=True)

def graph(method, path, body=None):
    """Microsoft Graph v1.0 as the Azure CLI user."""
    r = requests.request(method, f'https://graph.microsoft.com/v1.0{path}', json=body, timeout=60,
                         headers={'Authorization': f"Bearer {cli.get_token('https://graph.microsoft.com/.default').token}"})
    if not r.ok:
        raise RuntimeError(f'{method} {path}: HTTP {r.status_code} {r.text[:600]}')
    return r.json() if r.content else None

def call(method, url, scope, body=None):
    """ARM or the Foundry data plane with a CLI token for `scope`. The caller checks the response."""
    return requests.request(method, url, data=None if body is None else json.dumps(body), timeout=120,
                            headers={'Authorization': f'Bearer {cli.get_token(scope).token}', 'Content-Type': 'application/json'})

def az(*args):
    """Run an az command; return its JSON output, or raise with the end of stderr."""
    r = subprocess.run(['az', *args, '-o', 'json'], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"az {' '.join(args[:3])}: {r.stderr.strip()[-1500:]}")
    return json.loads(r.stdout) if r.stdout.strip() else None

def roles_of(principal_id):
    """The app roles a principal holds on news-mcp-{suffix} now, read from Graph."""
    held = {a['appRoleId'] for a in graph('GET', f'/servicePrincipals/{RESOURCE_SP_ID}/appRoleAssignedTo')['value']
            if a['principalId'] == principal_id}
    return [role for role in authz.ROLES if authz.permission_id(SUFFIX, role) in held]

print(f'Suffix          : {SUFFIX}')
print(f'Gateway         : {APIM_NAME} in {CORE_RG}')

Suffix          : c2676f
Gateway         : apim-foundry-c2676f in rg-foundry-core-c2676f


## Step 1: The MCP API on the gateway

[`infra/apim-news-mcp.bicep`](infra/apim-news-mcp.bicep) references `apim-foundry-{suffix}` as an existing resource and adds four things to it:

| Resource | Setting |
|---|---|
| named value `gnews-api-key` | secret: in the portal under **APIs** > **Named values**, with its value hidden. The policy reads it as `{{gnews-api-key}}` and sends it to GNews as `X-Api-Key` (see below) |
| backend `gnews-mcp` | `https://mcp.gnews.io/`, where GNews serves MCP. An APIM backend is a named record of a service that the gateway forwards requests to. It holds the service's address and how to reach it: its URL, plus optionally credentials, certificates, a proxy, or a circuit breaker and load-balancing settings. An API then refers to the backend by name instead of carrying the URL itself |
| API `news-mcp` | `type: mcp`, path `news-mcp`, `backendId` `gnews-mcp`, streamable transport, no subscription key |
| API policy | the policy file, with two placeholders filled in: the tenant id, and the two token audiences the policy accepts, which are the client id of `news-mcp-{suffix}` and its App ID URI `api://news-mcp-{suffix}`. A token for the news service carries one of them as `aud`: the client id in a v2 token, which is what the resource app asks for, or the URI in a v1 token |

**Why both a backend and an MCP server entry for one target?** APIM splits every API into two halves:

- **The MCP server entry (the API `news-mcp`)** is the front: what callers see. It owns the path on the gateway (`/news-mcp/mcp`), the type (`mcp`), the policy, the subscription setting, product membership and telemetry. Foundry's connection points at this.
- **The backend entry (`gnews-mcp`)** is the back: where the gateway sends the request. It owns the upstream URL and anything about reaching it (credentials, certificates, timeouts, circuit breaker, load balancing).

The named value appears at the end of the policy's inbound section, the last thing before the request is forwarded. The user's token is removed and the key takes its place:

```xml
<set-header name="Authorization" exists-action="delete" />
<set-header name="X-Api-Key" exists-action="override">
  <value>{{gnews-api-key}}</value>
</set-header>
```

The `backendId` is what makes the API a pass-through. Without it APIM answers `initialize` itself and lists the tools of the API's own operations, of which there are none. Clients reach the server at `/mcp` under the API's path, and APIM appends that `/mcp` to the backend URL; GNews has nothing there, so the policy ends with `rewrite-uri` to `/`.

The cell lists the gateway's APIs before and after, to show that the deployment added one and left the others as they were.

In [4]:
def apis():
    r = call('GET', f'{APIM}/apis?api-version={APIM_API_VERSION}', ARM)
    r.raise_for_status()
    return {a['name']: (a['properties'].get('type') or 'http', a['properties']['path']) for a in r.json()['value']}


before = apis()
params = {'suffix': SUFFIX, 'tenantId': TENANT_ID, 'audiences': [RESOURCE_APP_ID, authz.app_id_uri(SUFFIX)],
          'gnewsApiKey': os.environ['GNEWS_API_KEY']}
with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False) as f:
    json.dump({'$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
               'contentVersion': '1.0.0.0', 'parameters': {k: {'value': v} for k, v in params.items()}}, f)
t0 = time.time()
try:
    result = az('deployment', 'group', 'create', '-g', CORE_RG, '-n', 'news-mcp-apim',
                '--template-file', str(LAB_DIR / 'infra' / 'apim-news-mcp.bicep'), '--parameters', f'@{f.name}')
finally:
    os.unlink(f.name)
APIM_MCP_URL = result['properties']['outputs']['mcpUrl']['value']
print(f'Deployed in {time.time() - t0:.0f}s: {APIM_MCP_URL}\n')

api = call('GET', f'{APIM}/apis/news-mcp?api-version={APIM_API_VERSION}', ARM).json()['properties']
backend = call('GET', f"{APIM}/backends/{api['backendId']}?api-version={APIM_API_VERSION}", ARM).json()['properties']
print(f"news-mcp: type={api['type']}  path={api['path']}  backend={api['backendId']} ({backend['url']})  subscriptionRequired={api['subscriptionRequired']}")

after = apis()
print(f'\nAPIs on the gateway: {len(before)} before, {len(after)} after')
for name, (kind, path) in sorted(after.items()):
    state = 'added' if name not in before else 'unchanged' if before[name] == (kind, path) else 'CHANGED'
    print(f'  {name:<22} {kind:<5} /{path:<22} {state}')

Deployed in 37s: https://apim-foundry-c2676f.azure-api.net/news-mcp/mcp

news-mcp: type=mcp  path=news-mcp  backend=gnews-mcp (https://mcp.gnews.io/)  subscriptionRequired=False

APIs on the gateway: 2 before, 2 after
  news-mcp               mcp   /news-mcp               unchanged
  openai                 http  /openai                 unchanged


## Step 2: The gateway refuses a request without an Entra token

The gateway holds the GNews key, so it must not be an open door to GNews. The cell sends a `tools/list` request to the new API with no `Authorization` header at all, the way anyone on the internet could. `validate-azure-ad-token` is the first thing the policy does, so the request stops at the gateway with HTTP 401 and never reaches GNews.

From here on, only a request that carries a token for `news-mcp-{suffix}` gets past this point: the token Foundry obtains on the user's behalf before each tool call. A new API can take a few seconds to answer after the deployment; the cell waits for up to two minutes.

In [3]:
for attempt in range(13):
    r = requests.post(APIM_MCP_URL, json={'jsonrpc': '2.0', 'id': 1, 'method': 'tools/list'},
                      headers={'Accept': 'application/json, text/event-stream'}, timeout=30)
    if r.status_code == 401:
        break
    time.sleep(10)
print(f'tools/list without a token: HTTP {r.status_code}')
assert r.status_code == 401

tools/list without a token: HTTP 401


## Step 3: Give `news-desk-agent` an MCP tool that calls the gateway

`news-desk-agent` is the Foundry agent 08-05c-01 created. Its definition has three parts: the model (`CHAT_MODEL`, `gpt-5.4-mini` by default), the instructions, and the tools. Agents are versioned: each change to the definition adds a version. 08-05c-01 created the agent with no tools.

The cell first reads the `news-mcp-apim` connection from the project and checks that its target is this API's URL. It then creates a new version of `news-desk-agent` with one MCP tool, `news_mcp_apim`, that names that connection. This is the link 08-05c-01 Step 10 left for this notebook: from now on, when the model calls a news tool, Foundry sends the MCP request to the gateway with a token obtained on the user's behalf, as the connection's `AgentOnBehalfOfUser` says.

In [6]:
connection = call('GET', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, 'news-mcp-apim', CONNECTION_API), ARM).json()
print(f"Connection news-mcp-apim: authType={connection['properties']['authType']}  target={connection['properties']['target']}")
assert connection['properties']['target'] == APIM_MCP_URL

agent = project.agents.create_version(agent_name=ident.AGENT_NAME, definition=PromptAgentDefinition(
    model=CHAT_MODEL, instructions=ident.AGENT_INSTRUCTIONS,
    tools=[ident.mcp_tool('news_mcp_apim', APIM_MCP_URL, connection['id'])]))
print(f'{agent.name} v{agent.version}: tool news_mcp_apim -> {APIM_MCP_URL}')

Connection news-mcp-apim: authType=AgentOnBehalfOfUser  target=https://apim-foundry-c2676f.azure-api.net/news-mcp/mcp
news-desk-agent v7: tool news_mcp_apim -> https://apim-foundry-c2676f.azure-api.net/news-mcp/mcp


## Step 4: Sign in as user A and user B

Two device code sign-ins, as in 08-05c-01 and 08-05c-02. Each cell prints the account it expects and stops if the token belongs to someone else.

| Cell | Sign in as | Account | Method |
|---|---|---|---|
| first | user A | your admin account, the one `az login` uses | your usual password and multifactor method |
| second | user B | `news-sports-desk@{your tenant's default domain}`, in `.env` as `NEWS_TEST_USER_UPN` | Temporary Access Pass, in `.env` as `NEWS_TEST_USER_TAP` |

`grep -E 'NEWS_TEST_USER_(UPN|TAP)' .env` from the repository root shows user B's account and pass.

For user B:

1. Open a private browser window. A normal window offers the account you are already signed in with, and picking it signs in user A instead.
2. Open the address the cell prints and enter its code.
3. The page names `news-desk-client-{suffix}` and the location of the machine that runs the kernel. That is expected: continue.
4. If an account picker appears, choose **Use another account** and type user B's account.
5. If Entra asks for a password, choose **Other ways to sign in**, then **Temporary Access Pass**, and paste the pass.

The roles each user holds now come from Graph. If 08-05c-02 has run, user B has lost `News.Read.Science`; otherwise user B still holds it, and Step 5 expects accordingly.

In [7]:
print('User A, your admin account:')
cred_a = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)
token = cred_a.get_token(f'api://{BLUEPRINT_CLIENT_ID}/access_agent').token  # the sign-in; cred_a caches it
ADMIN_ID = checks.jwt_claims(token)['oid']
if ADMIN_ID == TEST_USER_ID:
    raise RuntimeError('That sign-in was user B. Sign in as user A, your admin account, and re-run this cell.')
print(f'\nSigned in as user A, who holds {roles_of(ADMIN_ID)}')

User A, your admin account:
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code ABD26STUZ to authenticate.

Signed in as user A, who holds ['News.Read.All']


In [8]:
print(f"User B, {os.environ['NEWS_TEST_USER_UPN']}, in a private window, with the Temporary Access Pass in .env (NEWS_TEST_USER_TAP):")
cred_b = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)
token = cred_b.get_token(f'api://{BLUEPRINT_CLIENT_ID}/access_agent').token  # the sign-in
if checks.jwt_claims(token)['oid'] != TEST_USER_ID:
    raise RuntimeError("That sign-in was not user B. A device code page reuses any account already signed in to the browser: open a private window, choose \"Use another account\", sign in as user B with the pass, and re-run this cell.")
print('Signed in as user B')

clients = {user: checks.agent_client(cred, PROJECT_ENDPOINT, ident.AGENT_NAME, BLUEPRINT_CLIENT_ID)
           for user, cred in (('A', cred_a), ('B', cred_b))}
ROLES = {'A': roles_of(ADMIN_ID), 'B': roles_of(TEST_USER_ID)}
print(f"\nUser B holds {ROLES['B']}")

User B, {user}@{tenant}.onmicrosoft.com, in a private window, with the Temporary Access Pass in .env (NEWS_TEST_USER_TAP):
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code ETBYBP2Y3 to authenticate.
Signed in as user B

User B holds ['News.Read.Sports', 'News.Read.Science', 'News.Read.Health']


## Step 5: The test matrix

Each case sends one question to the agent as one user and checks what comes back. Every question has the form "Give me three {category} headlines", so the model calls `top_headlines` with that category as its argument, using GNews's own category names. For each call the policy reads the tool name and the category from the request body, turns the `roles` in the token into the categories the user may read (`News.Read.All` allows all nine; `News.Read.Sports` allows `sports`, and so on), and refuses the call before it reaches GNews if the category is not among them. The cases come from [`news_mcp_checks.py`](news_mcp_checks.py):

| # | User | Asks for | Expected outcome | Why |
|---|---|---|---|---|
| 1 | A | business headlines | `articles`: three business headlines | user A holds `News.Read.All` |
| 2 | B | business headlines | `denied`: the denial text, which names user B's categories | user B has no business role, so the policy refuses the call and GNews is never called |
| 3 | B | sports headlines | `articles`: three sports headlines | user B holds `News.Read.Sports` |
| 4 | B | science headlines | `articles` if user B still holds `News.Read.Science`; `denied` if 08-05c-02 Step 8 revoked it | the expectation is set from the roles read from Graph at sign-in |

Both users see the same two tools, `top_headlines` and `search`. The gateway cannot filter that list (see the introduction), so the difference between the users shows only when a call is made.

What happens to the denial has differed between runs. On 2026-11-10 the tool result with `isError: true` went back to the model, which repeated the policy's text to the user, as the agent's instructions tell it to: the matrix line shows `called top_headlines` and `denied`, and the reply names user B's categories. On 2026-10-30 and 2026-11-06 Foundry instead failed the whole request with HTTP 400 `tool_user_error`, first with the policy's text in the message ("An error occurred invoking 'top_headlines': ...") and later with only "A user error occurred while executing a tool call." `checks.ask` and `checks.classify` handle all three: `denied` when the policy's text is in the call result or in the error, and `failed` when a tool error carries no text, which never passes because a denial cannot be told from any other failure. An application calling the agent must handle both a normal reply that quotes the denial and an HTTP 400 that may carry no text.

The cell prints a table with one row per case: the user, the category asked for, how many tools the agent listed, the tool it called, the outcome, the expected outcome, PASS or FAIL, and what the tool call returned: the denial text for a refused call, or the number of articles and the first headline for an allowed one. A legend under the table says what each outcome means. It stops with an error if a case did not get its expected outcome. The last lines show the tools each user saw and how many GNews requests the kernel has made so far.

In [9]:
science = 'articles' if 'News.Read.Science' in ROLES['B'] else 'denied'
print(f"User B holds {', '.join(ROLES['B'])}, so science should be {science}.\n")
cases = (*checks.base_cases('apim'), checks.science_case(science))
records = checks.run_matrix(clients, cases, ROLES)
print()
for user in ('A', 'B'):
    listed = next(r['listed'] for r in records if r['user'] == user and r['listed'])
    print(f"Tools listed to user {user}: {', '.join(listed)}")

User B holds News.Read.Sports, News.Read.Science, News.Read.Health, so science should be articles.

User  Asked for  Tools listed  Tool called             Outcome      Expected   Result  Tool result
----  ---------  ------------  ----------------------  -----------  ---------  ------  -----------
A     business              2  top_headlines           articles     articles   PASS    {"information":{"realTimeArticles":{"message":"On the Free plan, articles...
B     business              2  top_headlines           denied       denied     PASS    Your access covers: sports, science, health. Do not retry other categories.
B     sports                2  top_headlines           articles     articles   PASS    {"information":{"realTimeArticles":{"message":"On the Free plan, articles...
B     science               2  top_headlines           articles     articles   PASS    {"information":{"realTimeArticles":{"message":"On the Free plan, articles...

Outcomes
  articles     a call for the request

**Observed on 2026-10-30.** User B's business and science requests both came back as HTTP 400 `tool_user_error` carrying the policy's text, so no reply formed and the response had no items: that is why those two lines list no tools. Both users' other requests listed `search` and `top_headlines`, and user A's business call returned articles, so the policy's read of the body (`preserveContent: true`) did not stop it reaching GNews.

Two things the gateway needed that the published material did not say. At `2025-09-01-preview` the service rejected `mcpProperties.endpoints` as an array, the shape the REST spec and its example give, and asked for a dictionary; this lab uses `2024-06-01-preview` with a backend, as the AI-Gateway samples do. And an MCP API without `backendId` answered `initialize` as "Azure API Management" with no tools. While that first version was live, Foundry served user A an empty tool list, and kept doing so for user A's token after the API was fixed, even on a new agent version; a new sign-in cleared it.

**Observed on 2026-11-10.** All four cases passed. User B's business request came back as a normal response: the model called `top_headlines`, the tool result was the denial with `isError: true`, and the reply repeated the policy's text. Earlier that morning, from 10:18 to 10:31 UTC, every run had failed inside Foundry within a millisecond with `tool_user_error` and no request reached the gateway; the connection was stored as `AgentUserImpersonation` at the time (see the note under 08-05c-01 Step 10). After it read back as `AgentOnBehalfOfUser`, from 10:42 UTC, the runs succeeded.

## Your own questions

Ask the agent anything as either user: set `USER` and `QUESTION` and run the cell as often as you like. The cell prints the tools the agent saw, the calls it made and the reply. When the policy denies a call, the reply is the denial text (quoted by the model on 2026-11-10; in earlier runs Foundry returned an HTTP 400 instead, which the cell prints). A reply that fetches articles uses one GNews request.

To try on this track, with what to expect in the demo runbook of [08-05c-00](08-05c-00-news-mcp-per-user-access.md):

- `What news tools do you have?` as A, then as B: the same two tools for both
- `Give me three business headlines` as B: the denial text, in the reply or as an HTTP 400
- `Search for news about the Champions League` as B: the denial text, because `search` needs `News.Read.All`

In [12]:
USER = 'B'                                   # 'A' or 'B'
QUESTION = 'Give me three science headlines'

response = checks.ask(clients[USER], QUESTION)
print(f'Tools listed : {checks.listed_tools(response)}')
print(f'Tools called : {checks.called_tools(response)}')
print(f'Reply        : {checks.reply(response)}')

Tools listed : ['search', 'top_headlines']
Tools called : [('top_headlines', {'category': 'science', 'lang': 'en', 'max': 3, 'page': 1})]
Reply        : Here are three science headlines:

1. **Machine learning reveals how often fish schooling evolved-and repeatedly disappeared**
2. **Tin deposits are formed from source rocks, not cooling**
3. **The ELT can probe ultra-diffuse galaxies and their dark matter mysteries**

Note: the tool says free-plan articles arrive with a 12-hour delay.


## Cleanup

The full teardown is the Cleanup section of [08-05c-01](08-05c-01-identity-and-agent.ipynb), which removes this notebook's API, backend, named value and connection too. To remove only Track B, uncomment the cell: the API (its policy goes with it), the backend, the named value, and the `news-mcp-apim` connection. The `news-desk-agent` version that uses the connection stays until 08-05c-01 deletes the agent.

In [11]:
# for path in ('apis/news-mcp', 'backends/gnews-mcp', 'namedValues/gnews-api-key'):
#     print(path, call('DELETE', f'{APIM}/{path}?api-version={APIM_API_VERSION}', ARM).status_code)
# print('news-mcp-apim', call('DELETE', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, 'news-mcp-apim', CONNECTION_API), ARM).status_code)